In [1]:
import pandas as pd
import numpy as np

In [4]:
base_path = "Tofinal/"

df1 = pd.read_csv(base_path + "Personal_Finance_Dataset.csv")      # Dataset 1
df2 = pd.read_csv(base_path + "synthetic_expense_data_cleaned.csv")        # Dataset 2
df3 = pd.read_csv(base_path + "Daily Household Transactions.csv")  # Dataset 3
#df4 = pd.read_csv(base_path + "Expenses.csv")                      # Dataset 4 (PH expenses)
#df5 = pd.read_csv(base_path + "budjet.csv")                        # Dataset 5 (Lux budget)
#df6 = pd.read_csv(base_path + "expenses_income_summary.csv")       # Dataset 6 (INR mixed)
df7 = pd.read_csv(base_path + "transaction.csv")                   # Dataset 7 (whatever you cleaned there)

print("[INFO] Shapes:")
print("df1:", df1.shape)
print("df2:", df2.shape)
print("df3:", df3.shape)
#print("df4:", df4.shape)
#print("df5:", df5.shape)
#print("df6:", df6.shape)
print("df7:", df7.shape)


[INFO] Shapes:
df1: (1500, 10)
df2: (1000, 10)
df3: (2452, 10)
df7: (1040203, 10)


Standardize Schema per Dataset

In [5]:
CORE_COLS = [
    "Date",
    "Transaction Description",
    "Category",
    "Amount",
    "Type",
    "UserID",
    "TransactionID",
    "Currency",
    "Merchant",
    "Account Name",
    "SourceDataset",
]

def standardize_one(df, source_name):
    # 1) Rename common variants → our canonical names
    rename_map = {
        "Transaction_ID": "TransactionID",
        "Transaction Id": "TransactionID",
        "Account_Name": "Account Name",
        "Account": "Account Name",
        "AccountName": "Account Name",
        "Description": "Transaction Description",
        "TransactionDescription": "Transaction Description",
        "TransDescription": "Transaction Description",
        "Category": "Category",
        "Amount": "Amount",
        "Date": "Date",
        "Type": "Type",
        "UserID": "UserID",
        "User Id": "UserID",
        "Currency": "Currency",
        "Merchant": "Merchant",
    }
    df = df.rename(columns=rename_map)

    # 2) Make sure all core fields exist
    for col in CORE_COLS:
        if col not in df.columns:
            df[col] = np.nan  # start as NaN

    # 3) Add SourceDataset label
    df["SourceDataset"] = source_name

    # 4) Return only core cols (plus keep others if you want later)
    return df[CORE_COLS].copy()

Apply Schema Standardization to All 7

In [7]:
df1_std = standardize_one(df1, "personal_finance")
df2_std = standardize_one(df2, "synthetic_expenses")
df3_std = standardize_one(df3, "household_daily")
#df4_std = standardize_one(df4, "ph_expenses")
#df5_std = standardize_one(df5, "lux_budget")
#df6_std = standardize_one(df6, "inr_summary")
df7_std = standardize_one(df7, "unknown_transactions")

print("[INFO] Standardized shapes:")
for i, d in enumerate([df1_std, df2_std, df3_std, df7_std], start=1):
    print(f"df{i}_std:", d.shape)

[INFO] Standardized shapes:
df1_std: (1500, 11)
df2_std: (1000, 11)
df3_std: (2452, 11)
df4_std: (1040203, 11)


Global Cleaning Helpers

Date cleaning

In [8]:
def clean_dates(df):
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.date
    return df

Amount cleaning (including commas & dropping invalid / ≤ 0)

In [9]:
def clean_amount(df):
    # Convert everything to strings, remove thousand-separators commas
    df["Amount"] = (
        df["Amount"]
        .astype(str)
        .str.replace(",", "", regex=False)  # "3,800.00" -> "3800.00"
        .str.strip()
    )
    
    df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
    
    before = df.shape[0]
    df = df[df["Amount"].notna() & (df["Amount"] > 0)].copy()
    after = df.shape[0]
    print(f"[INFO] Dropped {before - after} rows due to invalid Amount.")
    return df

Text normalization & Type normalization

In [10]:
def clean_text_and_type(df):
    # Category / Merchant / Account / Description → basic cleaning
    for col in ["Category", "Merchant", "Account Name", "Transaction Description"]:
        df[col] = df[col].fillna("").astype(str).str.strip()
    
    # Standardize Type
    def normalize_type(t):
        t = str(t).strip().lower()
        if "exp" in t or "debit" in t:
            return "Expense"
        if "inc" in t or "credit" in t or "salary" in t or "income" in t:
            return "Income"
        if "transfer" in t:
            return "Transfer"
        return "Other"
    
    df["Type"] = df["Type"].apply(normalize_type)
    return df

Ontology Layer: UnifiedCategory

In [11]:
def map_unified_category(row):
    cat = str(row["Category"]).lower()
    desc = str(row["Transaction Description"]).lower()
    text = cat + " " + desc  # combined text

    # 🔹 Special "milk" case → treat as groceries / food
    if "milk" in text:
        return "Groceries & Household"

    # Food & Drinks
    food_keywords = [
        "food", "drink", "restaurant", "restuarant", "cafe",
        "coffee", "coffe", "juice", "snack", "market", "grocery",
        "eating", "lunch", "dinner", "breakfast"
    ]
    if any(k in text for k in food_keywords):
        return "Food & Drinks"

    # Transport
    transport_keywords = [
        "transport", "bus", "taxi", "train", "uber", "grab",
        "fuel", "gas", "tire", "parking", "commute"
    ]
    if any(k in text for k in transport_keywords):
        return "Transport"

    # Housing / Rent
    housing_keywords = ["rent", "apartment", "home", "mortgage", "house"]
    if any(k in text for k in housing_keywords):
        return "Housing"

    # Bills & Utilities
    bills_keywords = [
        "bill", "bills", "fees", "electricity", "water",
        "phone", "internet", "utility", "communal", "cable"
    ]
    if any(k in text for k in bills_keywords):
        return "Bills & Utilities"

    # Entertainment
    entertainment_keywords = [
        "movie", "cinema", "netflix", "hulu", "spotify",
        "entertainment", "events", "ticket", "concert", "joy",
        "film", "enjoyment"
    ]
    if any(k in text for k in entertainment_keywords):
        return "Entertainment"

    # Health
    health_keywords = [
        "health", "medical", "pharmacy", "hospital", "insurance",
        "doctor", "medicine"
    ]
    if any(k in text for k in health_keywords):
        return "Health"

    # Shopping
    shopping_keywords = [
        "shopping", "clothing", "shoes", "personal items",
        "tech", "electronics", "walmart", "mall"
    ]
    if any(k in text for k in shopping_keywords):
        return "Shopping"

    # Savings / Investment
    saving_keywords = [
        "savings", "investment", "pension", "fund", "fidelity",
        "mutual fund"
    ]
    if any(k in text for k in saving_keywords):
        return "Savings & Investment"

    # Debt / Fees
    debt_keywords = [
        "debt", "loan", "credit card", "repayment", "bank fees",
        "fee"
    ]
    if any(k in text for k in debt_keywords):
        return "Debt & Fees"

    # Income-specific (salary, refund, etc.)
    if row["Type"] == "Income":
        income_keywords = ["salary", "paycheck", "refund", "reward", "bonus"]
        if any(k in text for k in income_keywords):
            return "Income"

    # Transfers
    if row["Type"] == "Transfer":
        return "Transfer"

    # Default fallback
    return "Other"

In [12]:
def add_unified_category(df):
    df["UnifiedCategory"] = df.apply(map_unified_category, axis=1)
    return df

Apply Cleaning Pipeline to Each Standardized Dataset

In [13]:
def full_clean_pipeline(df, name):
    print(f"\n==== Cleaning {name} ====")
    df = clean_dates(df)
    df = clean_amount(df)
    df = clean_text_and_type(df)
    df = add_unified_category(df)
    print(f"[INFO] Done {name}: shape = {df.shape}")
    return df

df1_clean = full_clean_pipeline(df1_std, "personal_finance")
df2_clean = full_clean_pipeline(df2_std, "synthetic_expenses")
df3_clean = full_clean_pipeline(df3_std, "household_daily")
#df4_clean = full_clean_pipeline(df4_std, "ph_expenses")
#df5_clean = full_clean_pipeline(df5_std, "lux_budget")
#df6_clean = full_clean_pipeline(df6_std, "inr_summary")
df7_clean = full_clean_pipeline(df7_std, "unknown_transactions")


==== Cleaning personal_finance ====
[INFO] Dropped 0 rows due to invalid Amount.
[INFO] Done personal_finance: shape = (1500, 12)

==== Cleaning synthetic_expenses ====
[INFO] Dropped 0 rows due to invalid Amount.
[INFO] Done synthetic_expenses: shape = (1000, 12)

==== Cleaning household_daily ====
[INFO] Dropped 0 rows due to invalid Amount.
[INFO] Done household_daily: shape = (2452, 12)

==== Cleaning unknown_transactions ====
[INFO] Dropped 0 rows due to invalid Amount.
[INFO] Done unknown_transactions: shape = (1040203, 12)


Combine, Drop Duplicates, Shuffle

In [14]:
# Combine all cleaned datasets
df_final = pd.concat(
    [df1_clean, df2_clean, df3_clean, df7_clean],
    ignore_index=True
)

print("\n[INFO] Combined raw shape:", df_final.shape)

# Drop perfect duplicates on main transaction fields
dup_before = df_final.shape[0]
df_final = df_final.drop_duplicates(
    subset=["UserID", "Date", "Amount", "Category", "Transaction Description", "Type"]
)
dup_after = df_final.shape[0]
print(f"[INFO] Dropped {dup_before - dup_after} exact duplicate rows.")

# Shuffle for training
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print("[INFO] Final dataset shape:", df_final.shape)
df_final.head()



[INFO] Combined raw shape: (1045155, 12)
[INFO] Dropped 384 exact duplicate rows.
[INFO] Final dataset shape: (1044771, 12)


,Date,Transaction Description,Category,Amount,Type,UserID,TransactionID,Currency,Merchant,Account Name,SourceDataset,UnifiedCategory
0,2016-04-13,Bought vegetables and fruits,Groceries,9.40,Expense,US8055,TRX8_227498,NaN,,,unknown_transactions,Other
1,2019-01-02,Fuel purchase at local station,Motor/Travel,60.97,Expense,US8186,TRX8_759940,NaN,,,unknown_transactions,Transport
2,2013-11-04,Parking fee,Motor/Travel,223.55,Expense,US8114,TRX8_474070,NaN,,,unknown_transactions,Transport
3,2011-03-18,Household daily essentials,Groceries,8.96,Expense,US8060,TRX8_245895,NaN,,,unknown_transactions,Housing
4,2012-07-04,Concert or event payment,Entertainment,32.20,Expense,US8121,TRX8_512994,NaN,,,unknown_transactions,Entertainment


Save

In [15]:
import os

os.makedirs("Tofinal", exist_ok=True)
out_path = "Tofinal/final_personal_finance_dataset_v2.csv"

df_final.to_csv(out_path, index=False, encoding="utf-8")
print(f"\n✅ Final unified dataset saved to: {out_path}")



✅ Final unified dataset saved to: Tofinal/final_personal_finance_dataset_v2.csv
